# Week 12 In-Class Exercise: Neural Networks in PyTorch — Classifying Handwritten Digits
In lecture we saw how a neural network is a stack of simple layers trained by gradient descent, and in the demo we trained one on Fashion MNIST with PyTorch. In this exercise, you'll apply those **same PyTorch tools** to a dataset you've seen before:
> **Digits** — 1,797 images of handwritten digits (0-9), each an 8x8 grayscale image flattened into **64 features**. It ships with scikit-learn, so there is nothing to download.

Your job is to *use* the library, not to derive anything: build a network, train it, tune it, break it and fix it, and judge it on held-out data. You'll follow the same workflow as the demo:
1. Load the data and turn it into **tensors**
2. Build a network with `nn.Sequential`
3. Write the **training loop** and read the learning curves
4. Choose a **learning rate** by experiment
5. Compare **network sizes**
6. Run **your own experiment**: change one setting and predict the effect
7. **Debug** two broken training loops
8. Evaluate **once** on the test set and look at the mistakes

Work through the cells in order. Cells that say **TODO** need your code, and the hints show the PyTorch calls you need. Questions have a place for your answer. Ask for help if you are stuck for more than five minutes.

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week12/week12_inclass_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup
Run this cell to import the libraries and set the random seeds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', size=13)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=11)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)

## Step 1: Load the Data and Make Tensors (Provided)
Same recipe as the demo, on a smaller dataset:
- split into **train / validation / test** (60 / 20 / 20, stratified),
- scale pixel values from 0-16 down to **0-1**,
- turn everything into **tensors**: features as `float32`, labels as `long` (integer class indices, which is what the loss function expects).

This cell is provided. Just run it. It also defines two helpers, `make_loader` (mini-batches) and `plot_history` (learning curves), that you will reuse.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

digits = load_digits()
X_all, y_all = digits.data / 16.0, digits.target          # pixel values 0-16  ->  0-1

X_tmp, X_test, y_tmp, y_test = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=RANDOM_STATE)

to_X = lambda a: torch.tensor(a, dtype=torch.float32)
to_y = lambda a: torch.tensor(a, dtype=torch.long)
X_train, X_val, X_test = to_X(X_train), to_X(X_val), to_X(X_test)
y_train, y_val, y_test = to_y(y_train), to_y(y_val), to_y(y_test)

print('train', tuple(X_train.shape), '| val', tuple(X_val.shape), '| test', tuple(X_test.shape))
print('X dtype:', X_train.dtype, '| y dtype:', y_train.dtype, '| pixel range:', X_train.min().item(), 'to', X_train.max().item())

def make_loader(X, y, batch_size=32, shuffle=False):
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)

def plot_history(history, title=''):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
    a1.plot(history['loss'], label='train'); a1.plot(history['val_loss'], label='validation')
    a1.set_xlabel('epoch'); a1.set_ylabel('loss'); a1.legend(); a1.grid(alpha=.3)
    a2.plot(history['accuracy'], label='train'); a2.plot(history['val_accuracy'], label='validation')
    a2.set_xlabel('epoch'); a2.set_ylabel('accuracy'); a2.legend(); a2.grid(alpha=.3)
    fig.suptitle(title); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 10, figsize=(13, 1.7))
for ax, img, lab in zip(axes, X_train[:10], y_train[:10]):
    ax.imshow(img.view(8, 8), cmap='binary'); ax.set_title(int(lab)); ax.axis('off')
plt.show()

## Step 2: Build the Network
A neural network in PyTorch is a stack of layers. `nn.Sequential` chains them in order, `nn.Linear(a, b)` is a dense layer from `a` inputs to `b` outputs, and `nn.ReLU()` is the activation between them. Unlike Keras, you state each layer's **input** size yourself.

**Your turn!**
1. Build `model`: **64 inputs** -> `Linear(64, 64)` + ReLU -> `Linear(64, 32)` + ReLU -> `Linear(32, 10)` (ten outputs, one per digit). Call `torch.manual_seed(RANDOM_STATE)` first so everyone starts from the same weights.
2. Print the model and count its parameters in code.
3. Push one batch of 32 training images through it and print the output shape.

**Hint (from the demo):**
```python
torch.manual_seed(RANDOM_STATE)
model = nn.Sequential(
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 10),
)
sum(p.numel() for p in model.parameters())   # total number of weights and biases
out = model(X_train[:32])                     # one batch of 32 images
```

In [ ]:
# TODO: build `model`, print it, print the parameter count, and print the shape of the output for one batch of 32 images.
# Your code here:

**Question:** The network has several thousand parameters and we have about a thousand training images. What risk does that suggest, and what will you look at in Step 3 to see whether it happens? Also, what do the 10 numbers in each output row represent (they are *raw scores*, not probabilities)?

*Your answer:*

## Step 3: Train It (Write the Training Loop)
Keras's `fit()` is a loop you write yourself in PyTorch. The function below trains a model and records a history like Keras's `fit()`. The validation code and the bookkeeping are written for you. The **training step** inside the loop is missing.

**Your turn!** Fill in the `TODO`. Each mini-batch needs these five lines, in this order:

**Hint (from the demo):**
```python
logits = model(xb)               # forward pass
loss = loss_fn(logits, yb)       # compare with the true labels
optimizer.zero_grad()            # clear the old gradients
loss.backward()                  # backward pass: compute new gradients
optimizer.step()                 # gradient descent step
```

In [ ]:
def train(model, optimizer, epochs=30, batch_size=32):
    loss_fn = nn.CrossEntropyLoss()
    loader = make_loader(X_train, y_train, batch_size, shuffle=True)
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0.0, 0
        for xb, yb in loader:
            # TODO: the training step (the five lines in the hint above).
            # Your code here (you need to define `logits` and `loss`):

            total_loss += loss.item() * len(xb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
        history['loss'].append(total_loss / len(X_train))
        history['accuracy'].append(correct / len(X_train))

        model.eval()
        with torch.no_grad():                                   # evaluation: no gradients needed
            val_logits = model(X_val)
            history['val_loss'].append(loss_fn(val_logits, y_val).item())
            history['val_accuracy'].append((val_logits.argmax(dim=1) == y_val).float().mean().item())
    return history

Now train the model from Step 2 for 30 epochs with plain SGD (learning rate 0.1) and plot the curves with `plot_history`.

**Your turn!** Rebuild the model so training starts fresh (copy your Step 2 code), create `torch.optim.SGD(model.parameters(), lr=0.1)`, call `train`, and plot.

In [ ]:
# TODO: rebuild the model, create the SGD optimizer, train for 30 epochs, plot the history, and print the final train/validation accuracy.
# Your code here:

**Question:** Describe the learning curves. How large is the gap between the training and validation curves? Is the network overfitting, underfitting, or neither, and what is your evidence? Would training for more epochs help?

*Your answer:*

## Step 4: Choose a Learning Rate
The learning rate is "arguably the most important hyperparameter" (slide 101). Try five values, training a **fresh** model (same seed, same architecture) for 30 epochs each:

`lrs = [0.001, 0.01, 0.1, 0.3, 1.0]`

**Your turn!**
1. For each learning rate, build a fresh model (call `torch.manual_seed(RANDOM_STATE)` first so every run starts from the same weights), make an SGD optimizer with that rate, call `train`, and store the history in the dictionary `results[lr]`.
2. Plot every run's **validation accuracy** against the epoch on one axis (legend with the learning rates) and print each run's final validation accuracy.

**Hint:**
```python
results = {}
for lr in lrs:
    torch.manual_seed(RANDOM_STATE)
    m = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
    results[lr] = train(m, torch.optim.SGD(m.parameters(), lr=lr), epochs=30)
```

In [ ]:
lrs = [0.001, 0.01, 0.1, 0.3, 1.0]
results = {}
# TODO: train a fresh model for each lr and store its history in results[lr]
# Your code here:

# TODO: plot validation accuracy vs epoch for every lr on one axis and print the final validation accuracies
# Your code here:

**Question:** Describe what happens at the smallest and the largest learning rate, and say which one you would choose. How does this match slide 101's advice that a good rate sits just below the point where training becomes unstable?

*Your answer:*

## Step 5: How Big Should the Network Be?
Slide 99 and 100 say: start small, add layers or neurons until you start overfitting. Compare four architectures with the same training recipe (SGD, learning rate 0.1, 30 epochs):

| name | hidden layers |
|---|---|
| tiny | one layer of 8 units |
| small | one layer of 16 units |
| medium | two layers: 64 units, then 32 units (your Step 2 network) |
| large | two layers of 256 units |

**Your turn!** For each architecture build a fresh model (seed first), train it with `train(...)`, and collect its **number of parameters**, final **training accuracy** and final **validation accuracy** in a table (a list of dictionaries turned into a `pandas` DataFrame, or just printed lines).

**Hint:**
```python
import pandas as pd
sizes = {'tiny': [8], 'small': [16], 'medium': [64, 32], 'large': [256, 256]}
def build(hidden):
    layers, size = [], 64
    for h in hidden:
        layers += [nn.Linear(size, h), nn.ReLU()]
        size = h
    return nn.Sequential(*layers, nn.Linear(size, 10))
```

In [ ]:
import pandas as pd
sizes = {'tiny': [8], 'small': [16], 'medium': [64, 32], 'large': [256, 256]}

# TODO: write build(hidden), train one model per entry in `sizes`, and show a table with
#       parameters, final training accuracy and final validation accuracy.
# Your code here:

**Question:** Use the table. Does a bigger network always do better on the validation set? Where do you see signs of underfitting, and where do you see the training and validation accuracy start to separate? Which size would you pick and why?

*Your answer:*

## Step 6: Your Own Experiment: Predict, Then Test
In Steps 4 and 5 we told you which settings to vary. Now **you choose**. Pick **one** setting from the menu, *predict* in writing what it will do to the network, then run it and see whether you were right. Predicting first is the point: it shows you how well you understand what each setting does.

| setting | baseline (the Step 3 network) | try one of these |
|---|---|---|
| batch size | 32 | 8 or 128 |
| number of epochs | 30 | 5 or 100 |
| optimizer | SGD, lr = 0.1 | Adam with lr = 0.001, or Adam with lr = 0.01 |
| activation function | ReLU | `nn.Tanh` or `nn.Sigmoid` |

**Your turn!**
1. Pick **one** setting and **one** alternative value from the table. Change only that one thing.
2. In the **prediction cell** below, write what you expect to happen to the training accuracy, the validation accuracy, and how fast the network learns, and *why*. Do this **before you run any code**.
3. Train the baseline and your variant with `train(...)` (same seed, so they start from the same weights). Print the final training and validation accuracy of both and plot their validation-accuracy curves on one axis.

**Hint:**
```python
torch.manual_seed(RANDOM_STATE); base = make_net()
h_base = train(base, torch.optim.SGD(base.parameters(), lr=0.1), epochs=30, batch_size=32)

torch.manual_seed(RANDOM_STATE); alt = make_net(nn.Tanh)            # change the activation ...
h_alt = train(alt, torch.optim.SGD(alt.parameters(), lr=0.1),       # ... or the optimizer ...
              epochs=30, batch_size=32)                              # ... or epochs / batch_size
```
(`train` takes `epochs` and `batch_size`, and you pass in whichever optimizer you like, for example `torch.optim.Adam(alt.parameters(), lr=1e-3)`.)

*Your prediction (write it before you run the code): which setting did you pick, and what do you expect?*

In [ ]:
def make_net(activation=nn.ReLU):
    # Provided: the Step 2 network, with a choice of activation function
    return nn.Sequential(nn.Linear(64, 64), activation(), nn.Linear(64, 32), activation(), nn.Linear(32, 10))

# TODO: train the baseline and your one change with train(...); print both final train/validation accuracies
#       and plot the two validation-accuracy curves on one axis.
# Your code here:


**Question:** Was your prediction right? Describe what you saw in terms of the **training accuracy**, the **validation accuracy** and **how fast** the network learned. If the result surprised you, what do you think happened?

*Your answer:*

## Step 7: Debug Two Broken Training Loops
Each cell below runs **without an error**, yet gives a bad result. These are silent bugs, the worst kind. For each one:
1. run the cell and read the **symptom**,
2. find the bug,
3. **edit the code in that cell** to fix it and rerun it until it reaches the target.

Use `X_train`, `y_train`, `X_val`, `y_val` from Step 1 and the `make_loader` helper.

### Case A: the loss will not go down
**Target:** train and validation accuracy above 0.95. *Clue:* look at the last layer of the model and at the loss function. What does `CrossEntropyLoss` already do for you?

In [ ]:
def case_a():
    torch.manual_seed(RANDOM_STATE)
    model = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(),
                          nn.Linear(32, 10), nn.Softmax(dim=1))
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.CrossEntropyLoss()
    loader = make_loader(X_train, y_train, 32, shuffle=True)
    for epoch in range(30):
        total = 0.0
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * len(xb)
        if epoch % 10 == 0 or epoch == 29:
            print(f'  epoch {epoch:>2}  train loss {total / len(X_train):.3f}')
    with torch.no_grad():
        print(f'  train accuracy {(model(X_train).argmax(1) == y_train).float().mean():.3f}'
              f'   validation accuracy {(model(X_val).argmax(1) == y_val).float().mean():.3f}')
case_a()

*Your answer (Case A): what was the bug and what was the symptom?*

### Case B: the loss goes up
**Target:** train and validation accuracy above 0.95. *Clue:* compare this loop with the one you wrote in Step 3. What is missing from the training step?

In [ ]:
def case_b():
    torch.manual_seed(RANDOM_STATE)
    model = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.CrossEntropyLoss()
    loader = make_loader(X_train, y_train, 32, shuffle=True)
    for epoch in range(30):
        total = 0.0
        for xb, yb in loader:
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)
        if epoch % 10 == 0 or epoch == 29:
            print(f'  epoch {epoch:>2}  train loss {total / len(X_train):.3f}')
    with torch.no_grad():
        print(f'  train accuracy {(model(X_train).argmax(1) == y_train).float().mean():.3f}'
              f'   validation accuracy {(model(X_val).argmax(1) == y_val).float().mean():.3f}')
case_b()

*Your answer (Case B): what was the bug and what was the symptom?*

## Step 8: Evaluate Once on the Test Set
Slide 64: **do not tune on the test set.** You chose the learning rate and the architecture with the validation set. Now, **exactly once**, evaluate your Step 3 `model` on the test set and look at its mistakes.

**Your turn!**
1. Put the model in evaluation mode and compute the test predictions inside `torch.no_grad()`. Print the test accuracy.
2. Show the **confusion matrix** (rows = true digit, columns = predicted digit).
3. Show up to 8 misclassified test images with their true and predicted labels.

**Hint:**
```python
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
model.eval()
with torch.no_grad():
    test_pred = model(X_test).argmax(dim=1)
ConfusionMatrixDisplay.from_predictions(y_test.numpy(), test_pred.numpy())
wrong = (test_pred != y_test).nonzero().flatten()[:8]     # indices of mistakes
# X_test[k].view(8, 8) turns one image back into 8x8
```

In [ ]:
# TODO: test accuracy, confusion matrix, and up to 8 misclassified images.
# Your code here:

**Question:** Is the test accuracy close to the validation accuracy from Step 3? Look at the misclassified images and the confusion matrix. Would a person also find those digits hard to read, and which digits get confused?

*Your answer:*

## Reflection Questions
Answer each question in the cell below it (1-3 sentences each).

**Q1:** In scikit-learn you call `model.fit(X, y)`. In PyTorch you wrote the loop yourself. Name two things that loop does that `fit` hid from you, and say what went wrong in Step 7 when one of them was left out.

*Your answer:*

**Q2:** Look back at the learning-rate sweep (Step 4), the size comparison (Step 5) and your own experiment (Step 6). Which setting mattered most for this problem, and how can you tell from your results?

*Your answer:*

**Q3:** In earlier weeks you trained other classifiers (for example a random forest or an SVM) on image data. What did the neural network need from you that those models needed less of, and what did it give you in return?

*Your answer:*